# Module 3: Sorting Algorithms — Verified

**Prerequisites:** Modules 1–2

Sorting is a cornerstone of CS. In this module, we don't just *implement* sorting algorithms — we **prove** they are correct. In ACL2, a correct sorting algorithm must satisfy two properties:

1. **Ordered output:** The result is sorted (predicate `orderedp`)
2. **Permutation:** The output contains exactly the same elements as the input

We study four algorithms from the ACL2 community books (`books/sorting/`):
- Insertion sort (`isort.lisp`)
- Merge sort (`msort.lisp`)
- Bubble sort (`bsort.lisp`)
- Quicksort (`qsort.lisp`)

**Learning Objectives:**
1. Define and verify insertion sort
2. Understand the `perm` predicate for permutation equivalence
3. Implement and verify merge sort with a termination measure
4. Compare bubble sort and quicksort approaches
5. Prove that all sorting algorithms are equivalent

## 1. What Makes a Sort Correct?

A function `sort` is a correct sorting algorithm if for every input list `x`:

$$\text{orderedp}(\text{sort}(x)) \quad \wedge \quad \text{perm}(\text{sort}(x), x)$$

Let's define the building blocks.

In [ ]:
; An ordered list: each element ≤ the next
; Uses ACL2's built-in lexorder for comparison
(defun orderedp (x)
  (cond ((endp x) t)
        ((endp (cdr x)) t)
        (t (and (lexorder (car x) (cadr x))
                (orderedp (cdr x))))))

In [ ]:
; Test orderedp
(orderedp '(1 2 3 4 5))

In [ ]:
(orderedp '(1 3 2 4 5))

In [ ]:
; Element count — how many times does e appear in x?
(defun how-many (e x)
  (cond ((endp x) 0)
        ((equal e (car x)) (+ 1 (how-many e (cdr x))))
        (t (how-many e (cdr x)))))

In [ ]:
(how-many 'a '(a b a c a))

**Key insight:** Two lists are permutations of each other if and only if every element appears the same number of times in both. The `how-many` approach (from `books/sorting/convert-perm-to-how-many.lisp`) is often easier to work with than the recursive `perm` predicate.

## 2. Insertion Sort

*Source: `books/sorting/isort.lisp`*

Insertion sort works by repeatedly inserting each element into its correct position in an already-sorted list.

In [ ]:
; Insert element e into sorted list x
(defun insert-sort (e x)
  (if (endp x)
      (cons e x)
    (if (lexorder e (car x))
        (cons e x)
      (cons (car x)
            (insert-sort e (cdr x))))))

In [ ]:
; Sort by repeated insertion
(defun isort (x)
  (if (endp x)
      nil
    (insert-sort (car x)
                 (isort (cdr x)))))

In [ ]:
; Let's test it
(isort '(3 1 4 1 5 9 2 6))

### Proving Insertion Sort Correct

We need three theorems:

In [ ]:
; 1. The output is ordered
(defthm orderedp-insert-sort
  (implies (orderedp x)
           (orderedp (insert-sort e x))))

In [ ]:
(defthm orderedp-isort
  (orderedp (isort x)))

In [ ]:
; 2. The output is a proper list
(defthm true-listp-isort
  (true-listp (isort x)))

In [ ]:
; 3. Element counts are preserved
(defthm how-many-insert-sort
  (equal (how-many a (insert-sort e x))
         (if (equal a e)
             (+ 1 (how-many a x))
           (how-many a x))))

In [ ]:
(defthm how-many-isort
  (equal (how-many e (isort x))
         (how-many e x)))

With `orderedp-isort` and `how-many-isort`, we have a **complete correctness proof** for insertion sort. The output is sorted and contains exactly the same elements as the input.

## 3. Merge Sort

*Source: `books/sorting/msort.lisp`*

Merge sort splits the list into halves, sorts each recursively, and merges the results. The split uses `evens` and `odds` to avoid computing length.

### Termination Challenge
Unlike insertion sort (which recurs on `(cdr x)`), merge sort recurs on `(evens x)` and `(odds x)`. We need to prove these are **strictly smaller** than `x`.

In [ ]:
; Split into even-indexed and odd-indexed elements
(defun evens (x)
  (cond ((endp x) nil)
        ((endp (cdr x)) (list (car x)))
        (t (cons (car x) (evens (cddr x))))))

In [ ]:
; odds = evens of cdr
(defun odds (x)
  (evens (cdr x)))

In [ ]:
; Merge two sorted lists
(defun merge2 (x y)
  (declare (xargs :measure (+ (acl2-count x) (acl2-count y))))
  (if (endp x)
      y
    (if (endp y)
        x
      (if (lexorder (car x) (car y))
          (cons (car x) (merge2 (cdr x) y))
        (cons (car y) (merge2 x (cdr y)))))))

Notice the `:measure` declaration — `merge2` takes two arguments that both shrink, so we need to tell ACL2 that the *sum* of their sizes decreases.

In [ ]:
; Termination lemma: evens of a 2+ element list is smaller
(defthm acl2-count-evens-strong
  (implies (and (consp x) (consp (cdr x)))
           (< (acl2-count (evens x)) (acl2-count x)))
  :rule-classes :linear)

In [ ]:
; The merge sort function
(defun msort (x)
  (if (endp x)
      nil
    (if (endp (cdr x))
        (list (car x))
      (merge2 (msort (evens x))
              (msort (odds x))))))

In [ ]:
(msort '(3 1 4 1 5 9 2 6))

### Correctness of Merge Sort

In [ ]:
; The output is ordered
(defthm orderedp-merge2
  (implies (and (orderedp x) (orderedp y))
           (orderedp (merge2 x y))))

In [ ]:
(defthm orderedp-msort
  (orderedp (msort x)))

In [ ]:
; Element counts are preserved through merge
(defthm how-many-merge2
  (equal (how-many e (merge2 x y))
         (+ (how-many e x) (how-many e y))))

In [ ]:
; Key lemma: evens + odds reconstitute the original
(defthm how-many-evens-and-odds
  (implies (consp x)
           (equal (+ (how-many e (evens x))
                     (how-many e (evens (cdr x))))
                  (how-many e x))))

In [ ]:
(defthm how-many-msort
  (equal (how-many e (msort x))
         (how-many e x)))

## 4. Bubble Sort

*Source: `books/sorting/bsort.lisp`*

Bubble sort repeatedly passes through the list, swapping adjacent elements that are out of order. The challenge is proving **termination** — we need a measure that strictly decreases with each pass.

The book uses `bnext-size`: the total number of inversions (pairs where a smaller element follows a larger one).

In [ ]:
; One pass of bubble sort
(defun bnext (x)
  (declare (xargs :measure (len x)))
  (cond ((endp x) x)
        ((endp (cdr x)) x)
        ((lexorder (car x) (cadr x))
         (cons (car x) (bnext (cdr x))))
        (t (cons (cadr x)
                 (bnext (cons (car x) (cddr x)))))))

In [ ]:
; Count inversions as a termination measure
(defun how-many-smaller (e x)
  (cond ((endp x) 0)
        ((equal e (car x)) (how-many-smaller e (cdr x)))
        ((lexorder (car x) e) (+ 1 (how-many-smaller e (cdr x))))
        (t (how-many-smaller e (cdr x)))))

In [ ]:
(defun bnext-size (x)
  (cond ((endp x) 0)
        (t (+ (how-many-smaller (car x) (cdr x))
              (bnext-size (cdr x))))))

In [ ]:
; Bubble sort: iterate bnext until ordered
(defun bsort (x)
  (declare (xargs :measure (bnext-size x)))
  (if (orderedp x)
      x
    (bsort (bnext x))))

In [ ]:
(bsort '(3 1 4 1 5 9 2 6))

## 5. Quicksort

*Source: `books/sorting/qsort.lisp`*

Quicksort partitions the list around a pivot, sorts each partition, and concatenates. The `filter` function selects elements based on a comparison relation.

In [ ]:
; Ordering relation selector
(defun rel (fn i j)
  (case fn
    (LT  (and (lexorder i j) (not (equal i j))))
    (LTE (lexorder i j))
    (GT  (and (lexorder j i) (not (equal i j))))
    (otherwise (lexorder j i))))

In [ ]:
; Filter elements by relation to pivot
(defun qfilter (fn x e)
  (cond ((endp x) nil)
        ((rel fn (car x) e)
         (cons (car x) (qfilter fn (cdr x) e)))
        (t (qfilter fn (cdr x) e))))

In [ ]:
; Quicksort
(defun qsort (x)
  (cond ((endp x) nil)
        ((endp (cdr x)) (list (car x)))
        (t (append (qsort (qfilter 'LT (cdr x) (car x)))
                   (cons (car x)
                         (qsort (qfilter 'GTE (cdr x) (car x))))))))

In [ ]:
(qsort '(3 1 4 1 5 9 2 6))

## 6. All Sorts Are Equivalent

*Source: `books/sorting/equisort.lisp`*

Since each sorting algorithm produces an ordered list that is a permutation of the input, they must all produce the **same output** (assuming a total order). This is because:

> **Theorem:** If two lists are both ordered and have the same element counts, they are equal.

This means `isort`, `msort`, `bsort`, and `qsort` all compute the same function!

In [ ]:
; Two sorted lists with the same element counts are equal
(defthm orderedp-same-how-many-equal
  (implies (and (orderedp x)
                (orderedp y)
                (true-listp x)
                (true-listp y))
           (iff (equal x y)
                (equal (how-many-list x)
                       (how-many-list y)))))

The `equisort.lisp` book proves:
```
(defthm isort-is-msort
  (equal (isort x) (msort x)))

(defthm isort-is-qsort
  (equal (isort x) (qsort x)))
```

This is a profound result: the *specification* (ordered + permutation) uniquely determines the function.

## 7. Complexity Notes

| Algorithm | Time Complexity | Space | Key Insight |
|---|---|---|---|
| Insertion sort | $O(n^2)$ worst, $O(n)$ best | $O(1)$ | Simple, good for small/nearly-sorted |
| Merge sort | $O(n \log n)$ always | $O(n)$ | Divide-and-conquer, stable |
| Bubble sort | $O(n^2)$ worst and average | $O(1)$ | Simple but slow |
| Quicksort | $O(n \log n)$ average, $O(n^2)$ worst | $O(\log n)$ | Fast in practice |

## 8. Exercises

**Exercise 1:** Define a predicate `sortedp-rev` that checks if a list is sorted in reverse (descending) order. Prove that `(sortedp-rev (reverse (isort x)))` holds.

**Exercise 2:** The `merge2` function uses a two-argument measure `(+ (acl2-count x) (acl2-count y))`. Explain why `(acl2-count x)` alone would not suffice.

**Exercise 3:** Define `count-inversions` that counts the number of pairs `(i,j)` with `i < j` but `(nth i x) > (nth j x)`. Prove that `(count-inversions (isort x)) = 0`.

**Exercise 4 (Challenge):** Define a stable merge sort — one that preserves the relative order of equal elements. Prove stability as a formal theorem.

---
**Next:** [Module 4 — Data Structures](04_data_structures.ipynb) — Binary search trees and balanced trees, verified.